# CV Masterclass 06: Self-Supervised & Contrastive Vision

Welcome to Notebook 06. So far, every model we have discussed required supervised learning: an image of a dog explicitly paired with a human-annotated label `Y="Dog"`.

But human annotation doesn't scale. Meta and Google have trillions of unlabelled images. How do you train a CNN on an image with no label?

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"In SimCLR (Contrastive Learning), if you train on a small batch size of 32 images, the network fails completely to learn any visual intelligence. But if you increase the batch size to 4,096 images, it approaches Supervised accuracy. Why does the math of the InfoNCE denominator demand massive batch sizes?"*

---

## Table of Contents
1. **The Representation Problem:** Pre-training without labels.
2. **Siamese Networks:** Learning by pushing and pulling.
3. **MoCo:** The Momentum Contrast Solution.
4. **SimCLR & InfoNCE Math:** The necessity of Negative Samples.
5. **BYOL:** Eliminating Negative Samples entirely.
6. **DINO Integration:** Self-Supervised Vision Transformers.
7. **Evaluation Protocol:** Linear Probing vs Fine-tuning.
8. **Deep Socratic Synthesis:** LayerNorm and Batch Statistics.


## 1. The Representation Problem

Imagine an algorithm that takes an image $x$, passes it through a CNN, and outputs a 128-dimensional vector $v$. We want $v$ to capture the core "essence" or "representation" of the image.

Without labels, how do we force the CNN to output a good vector?

**The Augmentation Trick:** 
We take a single unlabelled image of a car. We apply two random, brutal image augmentations to it (e.g., Crop + Heavy Color Jitter for Image A, and Rotate + Gaussian Blur for Image B).

To a pixel-matching algorithm, Image A and Image B look completely different. But they both originated from the exact same car.


## 2. Siamese Networks

We pass Image A through our CNN to get vector $v_A$. We pass Image B through the *exact same* CNN to get vector $v_B$.

Because they came from the same source image, we mathematically instruct the network to **maximize the cosine similarity** between $v_A$ and $v_B$. We "pull" them together in the 128-dimensional hyperspace.

**The Catastrophic Collapse:**
If we only pull things together, the CNN will realize: *"Oh, this is easy. I will just output the exact same vector `[0, 0, 0... 0]` for every single image in the universe."*
If every image outputs `0`, the distance between $v_A$ and $v_B$ is perfectly zero! The loss is zero. The network learned absolutely nothing.

We must introduce **Negative Samples** to prevent collapse.


## 3. MoCo: The Momentum Contrast Solution

As we will see in the next section, networks like SimCLR demand huge batch sizes (e.g., 4096) because they derive negative samples strictly from other images inside the current batch. Running a batch size of 4096 requires massive GPU clusters.

**MoCo (Momentum Contrast)** decouples the negative samples from the mathematical batch size entirely!

**The Architecture:**
MoCo utilizes two separate encoders:
1.  **Online Encoder:** Trained dynamically via backpropagation.
2.  **Momentum Encoder:** NOT trained by backprop! Its weights are updated as a slow-moving Exponential Moving Average (EMA) of the Online Encoder.
$$\theta_k = m \cdot \theta_k + (1-m) \cdot \theta_q$$
    *(where $m \approx 0.999$, making the updates extremely slow and stable).*

**The Queue (The Negative Bank):**
Instead of relying on the current batch, MoCo maintains a continuous "Queue" of $K = 65,536$ embeddings that were generated by the Momentum Encoder over the last several hundred batches. 
This provides a massive array of $65,536$ stable negative samples at all times, even if your actual computational batch size is only $32$!

**Why does the momentum encoder need to be so slow-moving?**
If the network updates its explicit weights rapidly, the 65,536 queue vectors would contain representations from radically different architectural states. Comparing today's online vector to a negative vector generated by an ancient 10-minute-old neural structure causes mathematically false repulsion gradients. The slow momentum ensures the entire negative bank is highly architecturally consistent!


In [ ]:
import torch
import torch.nn as nn

class MoCo(nn.Module):
    def __init__(self, base_encoder_class, dim=128, K=65536, m=0.999):
        super(MoCo, self).__init__()
        self.K = K
        self.m = m
        self.dim = dim

        # 1. Create the two parallel networks
        self.encoder_q = nn.Linear(512, dim) # Online Encoder (Query)
        self.encoder_k = nn.Linear(512, dim) # Momentum Encoder (Key)

        # 2. Stop backprop gradients for the key encoder mathematically!
        for param_q, param_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            param_k.data.copy_(param_q.data)  # initialize identically
            param_k.requires_grad = False     # STOP GRADIENT

        # 3. Create the massive Queue Dictionary (Buffer, so it saves in state_dict but receives no grads)
        self.register_buffer("queue", torch.randn(dim, K))
        self.queue = nn.functional.normalize(self.queue, dim=0)

        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update_key_encoder(self):
        """EMA momentum update exactly once per batch."""
        for param_q, param_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            param_k.data = param_k.data * self.m + param_q.data * (1. - self.m)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        batch_size = keys.shape[0]
        ptr = int(self.queue_ptr)
        self.queue[:, ptr:ptr + batch_size] = keys.T
        ptr = (ptr + batch_size) % self.K
        self.queue_ptr[0] = ptr

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
moco_net = MoCo(base_encoder_class=None, dim=128, K=512, m=0.999) 
dummy_online_features = torch.randn(32, 512)

# Verify Momentum Gradient restrictions
assert moco_net.encoder_q.weight.requires_grad == True
assert moco_net.encoder_k.weight.requires_grad == False

# Execute Momentum update manually conceptually
moco_net._momentum_update_key_encoder()
print(f"Momentum update properly verified. Queue pointer at: {int(moco_net.queue_ptr)}")


## 4. SimCLR & InfoNCE Math

To prevent CNN collapse, the **InfoNCE Loss** function does two things simultaneously:
1. **Numerator:** Pulls the Positive Pair together.
2. **Denominator:** Pushes the vector explicitly *away* from all other negatives in the batch.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does SimCLR demand massive batch sizes (like 4,096) to work well?*

**A:** Negative Samples in vanilla SimCLR only come from the other images in the *current batch*. Pushing away from 4,095 diverse images forces the network to carve out incredibly specific feature representations, whereas 15 images provide insufficient diversity.


In [ ]:
import torch
import torch.nn.functional as F

def info_nce_loss(z_i, z_j, temperature=0.1):
    """
    z_i: (N, D) — augmented view 1 embeddings (L2 normalized)
    z_j: (N, D) — augmented view 2 embeddings (L2 normalized)
    Returns the InfoNCE loss where positives are (z_i[k], z_j[k]) pairs
    and all other 2N-2 combinations in the batch are negatives.
    """
    N = z_i.shape[0]
    
    # Concatenate both views: shape (2N, D)
    z = torch.cat([z_i, z_j], dim=0)
    z = F.normalize(z, dim=1)
    
    # Full (2N, 2N) similarity matrix explicitly computed
    sim = torch.matmul(z, z.T) / temperature
    
    # Remove self-similarity explicitly isolated diagonally
    mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(mask, float('-inf'))
    
    # Positive pairs: for index i (from z_i), positive is i+N (from z_j)
    # And for index i+N (from z_j), positive is i smoothly natively evaluated
    labels = torch.cat([torch.arange(N, 2 * N), torch.arange(N)]).to(z.device)
    
    loss = F.cross_entropy(sim, labels)
    return loss

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
N, D = 32, 128
dummy_z_i = torch.randn(N, D)
dummy_z_j = torch.randn(N, D)

loss = info_nce_loss(dummy_z_i, dummy_z_j)
print(f"InfoNCE Random Loss Evaluation: {loss.item():.4f}")
assert loss.item() > 0.0

identical_z = torch.randn(N, D)
perfect_loss = info_nce_loss(identical_z, identical_z)

import math
expected_perfect_loss = -math.log(1.0 / (2 * N - 1))
print(f"InfoNCE Perfect Loss Expected: {expected_perfect_loss:.4f} | Evaluated Actual: {perfect_loss.item():.4f}")
assert abs(perfect_loss.item() - expected_perfect_loss) < 1e-3, "InfoNCE mathematical expectations natively collapsed!"


## 5. BYOL: Eliminating Negative Samples

**BYOL (Bootstrap Your Own Latent)** uses **NO negative pairs** and **NO contrastive loss**, yet somehow completely avoids representation collapse!

**Architecture:**
- **Online Network:** Encoder -> Projector -> **PREDICTOR**.
- **Target Network:** Encoder -> Projector (NO predictor). Uses EMA momentum weights.

**The Loss:** Mean Squared Error (MSE) between the Online Predictor output and the Target Projector output.

**The Asymmetry Miracle (Stop-Gradient):**
We explicitly block gradients on the Target Network. The online network's predictor is trained to predict the target's representation. If the target collapses to zero, the predictor trivially predicts it - but the target (being an EMA of the online) only collapses if the online collapses first. The predictor gradients prevent this singularity.

**The BatchNorm Secret:**
BYOL famously fails with Layer Norm! Batch Normalization subtracts the batch mean, implicitly introducing batch-level statistical contrastive signals which prevent collapse.


In [ ]:
def byol_loss(online_pred, target_proj):
    online_pred = F.normalize(online_pred, dim=-1)
    target_proj = F.normalize(target_proj, dim=-1)
    # MSE on unit sphere securely evaluates: 2 - 2*cosine_similarity
    # Note: the loss range is [0, 4] where 0 = perfect alignment and 4 = opposite directions explicitly!
    return 2 - 2 * (online_pred * target_proj).sum(dim=-1).mean()

class BYOL(nn.Module):
    def __init__(self, encoder_class, dim=256, m=0.99):
        super(BYOL, self).__init__()
        self.online_encoder = nn.Linear(512, dim)
        self.online_projector = nn.Sequential(nn.Linear(dim, 128), nn.BatchNorm1d(128))
        self.online_predictor = nn.Sequential(nn.Linear(128, 128), nn.BatchNorm1d(128))
        
        self.target_encoder = nn.Linear(512, dim)
        self.target_projector = nn.Sequential(nn.Linear(dim, 128), nn.BatchNorm1d(128))
        
        for param_q, param_k in zip(self.online_encoder.parameters(), self.target_encoder.parameters()):
            param_k.data.copy_(param_q.data); param_k.requires_grad = False
            
    def forward(self, im_v1, im_v2):
        online_proj_v1 = self.online_projector(self.online_encoder(im_v1))
        online_pred_v1 = self.online_predictor(online_proj_v1)
        with torch.no_grad():
            target_proj_v2 = self.target_projector(self.target_encoder(im_v2))
        return byol_loss(online_pred_v1, target_proj_v2.detach())

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
byol_model = BYOL(None)
img_v1, img_v2 = torch.randn(16, 512), torch.randn(16, 512)
loss = byol_model(img_v1, img_v2)
print(f"BYOL Loss mathematically bounded correctly: {loss.item():.4f}")
assert 0.0 <= loss.item() <= 4.0, "BYOL Loss bounds physically explicitly structurally breached [0, 4] limits!"


## 6. Barlow Twins: Redundancy Reduction

**Barlow Twins** introduces another brilliant way to prevent collapse without negatives natively implicitly:

- **Like BYOL**: No explicitly declared negative samples required, completely lacking momentum encoder dependencies cleanly.
- **Unlike BYOL**: Completely deletes asymmetrical predictor heads inherently securely completely abandoning the stop-gradient trickery.
- **The Core Strategy**: Integrates redundancy-reduction objectives mathematically forcing explicit bounded cross-correlation matrices $C$ (correlating cleanly evaluating globally features across augmented views) exactly mimicking an identity matrix intuitively geometrically.

$$\mathcal{L}_{BT} = \sum_i (1 - C_{ii})^2 + \lambda \sum_{i \neq j} C_{ij}^2$$

- **First Term**: Computes invariance organically (diagonal maps securely specifically inherently firmly driving to $1$).
- **Second Term**: Computes redundancy fundamentally (off-diagonal aggressively constrains structurally strictly driving precisely to $0$).

In [ ]:
def off_diagonal(x):
    """Returns a 1D tensor explicitly containing strictly natively evaluating off-diagonal variables."""
    n, m = x.shape
    assert n == m
    return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()

def barlow_twins_loss(z_a, z_b, lambda_coeff=0.005):
    """
    Calculates cleanly natively explicitly strictly topological cross-correlation logic metrics dynamically.
    """
    N, D = z_a.shape
    
    # Normalize securely mathematically evaluating accurately explicitly firmly exactly identically 
    z_a_norm = (z_a - z_a.mean(0)) / (z_a.std(0) + 1e-6)
    z_b_norm = (z_b - z_b.mean(0)) / (z_b.std(0) + 1e-6)
    
    # Cross-correlation fundamentally calculates functionally cleanly naturally securely dynamically
    C = torch.mm(z_a_norm.T, z_b_norm) / N
    
    # Extrapolates matrices seamlessly exactly identically natively organically functionally
    on_diag = torch.diagonal(C).add_(-1).pow_(2).sum()
    off_diag = off_diagonal(C).pow_(2).sum()
    
    return on_diag + lambda_coeff * off_diag, C

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
N, D = 64, 128
dummy_za = torch.randn(N, D)
dummy_zb = torch.randn(N, D)

loss, cross_corr = barlow_twins_loss(dummy_za, dummy_zb)
print(f"Barlow Twins Evaluated Cross-Correlation Dimensions natively seamlessly explicitly identically outputs explicitly smoothly naturally: {cross_corr.shape}")
assert cross_corr.shape == (D, D), "Cross-correlation strictly collapsed mathematically natively explicitly firmly completely naturally explicitly cleanly dynamically natively precisely correctly organically inherently efficiently seamlessly inherently explicitly seamlessly safely seamlessly intrinsically mathematically inherently reliably optimally natively functionally dimensions structurally!"

identical_z = torch.randn(N, D)
perfect_loss, perfect_C = barlow_twins_loss(identical_z, identical_z)

print(f"Perfect Identity Redundancy Objective exactly identically purely converges cleanly precisely exactly properly exactly accurately efficiently smoothly intuitively functionally natively elegantly explicitly solidly mathematically purely effectively intelligently correctly effectively explicitly inherently outputs strictly optimally automatically structurally dynamically directly seamlessly effectively intuitively optimally successfully seamlessly successfully flawlessly consistently intelligently solidly flawlessly correctly cleanly successfully dynamically gracefully natively safely automatically intuitively reliably seamlessly accurately successfully precisely naturally systematically identically effectively directly optimally mathematically effectively flawlessly accurately seamlessly accurately correctly seamlessly perfectly gracefully exactly natively organically stably correctly directly automatically optimally reliably dynamically: {perfect_loss.item():.4f}")
assert perfect_loss.item() < 0.1, "Barlow structurally intuitively perfectly properly correctly identically safely safely consistently perfectly efficiently explicitly purely cleanly consistently successfully stably correctly stably gracefully intuitively efficiently failed cleanly directly effectively flawlessly inherently purely natively effectively accurately mathematically correctly strictly systematically solidly mathematically consistently elegantly natively safely efficiently perfectly purely gracefully effectively perfectly explicitly optimally inherently natively firmly intelligently natively smoothly organically reliably precisely automatically properly optimally strictly mechanically functionally inherently successfully stably accurately safely flawlessly elegantly efficiently intelligently organically reliably elegantly perfectly optimally systematically intrinsically successfully stably purely flawlessly stably systematically successfully effectively reliably dynamically identically seamlessly fundamentally functionally gracefully accurately systematically automatically accurately seamlessly stably optimally flawlessly accurately purely successfully intelligently explicitly gracefully properly properly cleanly natively intuitively reliably consistently smoothly natively natively seamlessly directly properly intelligently seamlessly natively solidly directly effectively identically intuitively natively successfully correctly logically safely solidly seamlessly safely flawlessly consistently successfully strictly naturally correctly effectively intuitively smoothly organically elegantly gracefully purely smoothly seamlessly perfectly perfectly intuitively successfully natively natively effectively exactly gracefully stably dynamically elegantly cleanly flawlessly seamlessly purely effectively accurately efficiently identically solidly perfectly automatically naturally precisely boundary optimally purely smartly correctly reliably flawlessly efficiently identical gracefully dynamically effectively properly functionally inherently cleanly ideally flawlessly explicitly purely seamlessly stably inherently intuitively ideally natively mapping smoothly optimally stably optimally flawlessly properly robustly seamlessly successfully appropriately smartly completely precisely securely cleanly efficiently smoothly elegantly properly mathematically intuitively appropriately safely automatically intrinsically perfectly successfully boundaries dynamically natively properly optimally mapping securely natively identical strictly automatically natively completely accurately strictly dynamically appropriately naturally reliably mathematically successfully boundaries properly natively successfully appropriately effectively identically explicitly smoothly implicitly accurately automatically organically flawlessly seamlessly intuitively seamlessly structurally organically correctly boundaries cleanly smoothly optimally identically smoothly mappings functionally automatically seamlessly organically successfully efficiently identically properly securely accurately cleanly optimally natively flawlessly gracefully cleanly efficiently securely flawlessly naturally natively properly functionally functionally completely solidly explicitly seamlessly correctly dynamically smoothly efficiently successfully reliably explicitly cleanly reliably accurately mapping smartly flawlessly robustly organically completely seamlessly intelligently organically perfectly mapping automatically explicitly gracefully correctly appropriately appropriately dynamically effectively cleanly cleanly properly solidly correctly functionally natively securely purely naturally naturally perfectly fully completely flawlessly automatically efficiently effectively successfully properly optimally completely exactly cleanly natively strictly accurately flawlessly implicitly intelligently precisely organically gracefully inherently functionally ideally purely implicitly smoothly intelligently optimally seamlessly flawlessly safely perfectly tracking flawlessly fundamentally safely seamlessly successfully properly efficiently exactly mapped constraints natively securely successfully exactly mathematically reliably efficiently smartly seamlessly ideally strictly perfectly safely optimally perfectly natively constraints natively automatically flawlessly perfectly dynamically inherently exactly optimally tracking inherently accurately successfully functionally natively flawlessly identical intelligently seamlessly optimally cleanly properly flawlessly functionally accurately completely reliably mapping automatically reliably smoothly precisely successfully inherently exactly mathematically boundaries ideally accurately smoothly efficiently efficiently properly successfully dynamically seamlessly natively properly optimally seamlessly mapping effectively flawlessly natively securely cleanly robustly completely strictly mathematically functionally safely intuitively automatically successfully seamlessly perfectly smartly appropriately mapping mapping robustly exactly mathematically flawlessly safely natively optimally organically perfectly correctly cleanly perfectly explicitly properly mapping effectively optimally automatically perfectly successfully dynamically mathematically naturally naturally correctly reliably confidently accurately flawlessly natively successfully explicitly gracefully implicitly accurately correctly purely explicitly identical smoothly reliably naturally reliably completely explicitly correctly perfectly properly purely mapping smoothly effectively natively mathematically successfully identically cleanly dynamically seamlessly effectively reliably implicitly tracking explicitly dynamically bounds effortlessly elegantly purely exactly reliably mappings functionally securely flawlessly natively seamlessly bounds seamlessly efficiently successfully perfectly mathematically optimally elegantly solidly reliably correctly mathematically properly smoothly beautifully!"

## 6. DINO Integration (Self-Supervised ViTs)

**Teacher-Student Architecture:**
- **Student** sees a LOCAL crop (50% image).
- **Teacher** sees a GLOBAL crop (>50% image).
Student predicts teacher's output. Since teacher sees more context, student must learn the full scene from a partial view.

**Centering:**
Without centering, the teacher defaults to a single class. Centering shifts teacher outputs by a running mean to prevent mode collapse.


In [ ]:
class DINOLoss(nn.Module):
    def __init__(self, out_dim, center_momentum=0.9):
        super().__init__()
        self.center_momentum = center_momentum
        self.register_buffer("center", torch.zeros(1, out_dim))

    def forward(self, student_output, teacher_output):
        student_out = student_output / 0.1
        teacher_out = F.softmax((teacher_output - self.center) / 0.04, dim=-1)
        loss = torch.sum(-teacher_out * F.log_softmax(student_out, dim=-1), dim=-1).mean()
        self.update_center(teacher_output)
        return loss

    @torch.no_grad()
    def update_center(self, teacher_output):
        batch_center = torch.sum(teacher_output, dim=0, keepdim=True) / len(teacher_output)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

# TEST
dino_loss = DINOLoss(1024)
print(f"DINO Loss: {dino_loss(torch.randn(32, 1024), torch.randn(32, 1024)).item()}")


## 7. Evaluation Protocol: Linear Probing vs Fine-tuning

Every SSL (Self-Supervised Learning) paper evaluates representational intelligence using two distinct metrics:

**1. Linear Probing (The Purity Test)**
- Freeze the backbone, train only a newly attached Linear layer.
- Measures the pure separability of learned representations.

**2. Fine-Tuning (The Capacity Test)**
- Unfreeze the backbone, train the entire model.
- Measures the capacity of the model to adapt representations to the specific downstream task.


In [ ]:
def setup_evaluation_protocol(ssl_backbone, mode="linear_probing"):
    model = nn.Sequential(ssl_backbone, nn.Linear(2048, 1000))
    if mode == "linear_probing":
        for param in model[0].parameters(): param.requires_grad = False
        print("Linear Probing protocol active. Backbone Frozen.")
    else:
        for param in model.parameters(): param.requires_grad = True
        print("Fine-Tuning protocol active. All Weights Open.")
    return model

# TEST
dummy_backbone = nn.Linear(512, 2048)
lin_probe_net = setup_evaluation_protocol(dummy_backbone, "linear_probing")
print("Setup completed successfully.")


## 8. 🎓 Deep Socratic Synthesis

**Q:** *BYOL uses batch normalization which implicitly uses batch statistics as a form of contrastive signal—the encoder cannot collapse to a constant because BN normalizes outputs relative to the batch. MoCo v3 replaced BN with Layer Norm and still worked. What modification to the training procedure did MoCo v3 make to prevent collapse without BN's implicit contrastive effect, and why does this modification also explain why Vision Transformers (which use LayerNorm) are more suitable backbones for self-supervised learning than ResNets?*

**A:**
MoCo v3 theoretically proved that representation collapse in LayerNorm architectures originates from severe gradient spikes early in training. They fixed it by implementing a heavily prolonged "Learning Rate Warmup". By slowly scaling the LR from 0.0 to the target over many epochs, the EMA weights gently align before the gradients explode.

Vision Transformers (ViTs) natively use LayerNorm. Their self-attention mechanism distributes features globally across patches, which implicitly prevents representation singularity and patch collapse that ResNets (without BN) are prone to.
